# Exploratory Data Analysis: Marketing Campaign Performance
This notebook analyzes campaign performance, compares channels, identifies high ROI campaigns, and detects underperforming ones.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_theme(style="whitegrid")

In [ ]:
# Load Data
df = pd.read_csv('../data/processed/cleaned_marketing_data.csv')
df['Date'] = pd.to_datetime(df['Date'])
df.head()

## 1. Performance Trends Over Time

In [ ]:
# Monthly Trend of Spend vs Revenue
monthly_trend = df.groupby(df['Date'].dt.to_period('M'))[['Cost', 'Revenue']].sum().reset_index()
monthly_trend['Date'] = monthly_trend['Date'].astype(str)

plt.figure(figsize=(12, 6))
plt.plot(monthly_trend['Date'], monthly_trend['Revenue'], marker='o', label='Revenue', linewidth=2)
plt.plot(monthly_trend['Date'], monthly_trend['Cost'], marker='o', label='Cost', linewidth=2)
plt.title('Monthly Campaign Spend vs Revenue', fontsize=16)
plt.xlabel('Month', fontsize=12)
plt.ylabel('Amount ($)', fontsize=12)
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

## 2. Compare Channels

In [ ]:
# Bar chart for Channel comparison (ROI and Spend)
channel_perf = df.groupby('Channel').agg({'Cost': 'sum', 'Revenue': 'sum'}).reset_index()
channel_perf['ROI'] = ((channel_perf['Revenue'] - channel_perf['Cost']) / channel_perf['Cost']) * 100

fig, ax1 = plt.subplots(figsize=(10, 6))

sns.barplot(x='Channel', y='Cost', data=channel_perf, color='lightblue', label='Total Spend', ax=ax1)
ax1.set_ylabel('Total Spend ($)')

ax2 = ax1.twinx()
sns.lineplot(x='Channel', y='ROI', data=channel_perf, color='red', marker='o', label='ROI (%)', ax=ax2, linewidth=2)
ax2.set_ylabel('ROI (%)')

plt.title('Channel Comparison: Spend vs ROI', fontsize=16)
fig.legend(loc='upper right', bbox_to_anchor=(0.9, 0.9))
plt.show()

## 3. High ROI vs Underperforming Campaigns

In [ ]:
# Scatter plot to detect high/low returning campaigns
campaign_roi = df.groupby('Campaign_ID').agg({'Cost': 'sum', 'Revenue': 'sum'}).reset_index()
campaign_roi['Cost'] = campaign_roi['Cost'].replace(0, 0.01)  # avoid div zero
campaign_roi['ROI'] = ((campaign_roi['Revenue'] - campaign_roi['Cost']) / campaign_roi['Cost']) * 100

plt.figure(figsize=(10, 6))
sns.scatterplot(x='Cost', y='Revenue', hue='ROI', palette='coolwarm', data=campaign_roi, alpha=0.7)
plt.title('Campaign Spend vs Revenue (Color = ROI%)', fontsize=16)
plt.xlabel('Total Spend ($)')
plt.ylabel('Total Revenue ($)')

# Add threshold line (Break-even where Revenue = Cost)
max_val = min(campaign_roi['Cost'].max(), campaign_roi['Revenue'].max())
plt.plot([0, max_val], [0, max_val], 'k--', lw=2, label='Break-even (ROI=0)')
plt.legend()
plt.show()

## 4. Correlation Analysis

In [ ]:
# Heatmap for correlation
numeric_cols = df.select_dtypes(include=[np.number])
corr = numeric_cols.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1)
plt.title('Correlation Heatmap of Marketing Metrics', fontsize=16)
plt.show()